# Web-Gold-40K manual review workspace

This interactive CPU notebook is for Reviewer A or Reviewer B. It displays complete train/validation trajectories, invalid bbox evidence, and causal recovery pairs. Decisions are appended to an immutable CSV event log under `/kaggle/working`; source JSON and images are never edited.

Before starting:

1. Read `docs/DATASET_MANUAL_REVIEW_GUIDE.md` completely.
2. Attach `kiyasmahmud/web-gold-40k` and the latest output of `kiyasmahmud/web-gold-existing-data-improvement`.
3. Use CPU and an interactive **Edit** session, not a committed batch run.
4. Set your reviewer ID in cell 2. Reviewer A and B must use separate notebook sessions.
5. To resume a queue, attach its previously exported event CSV as an input; the notebook restores exactly one matching log automatically.

In [ ]:
# 1. Pull the reviewed Code branch and install the local package only.
from pathlib import Path
import subprocess
import sys

REPOSITORY = 'https://github.com/Kiyas-Mahmud/webagent.git'
REPO_ROOT = Path('/kaggle/working/webagent')
if (REPO_ROOT / '.git').is_dir():
    subprocess.run(['git', '-C', str(REPO_ROOT), 'pull', '--ff-only', 'origin', 'Code'], check=True)
else:
    subprocess.run(['git', 'clone', '--branch', 'Code', '--single-branch', REPOSITORY, str(REPO_ROOT)], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', '-e', str(REPO_ROOT)], check=True)
sys.path.insert(0, str(REPO_ROOT / 'scripts'))
print('Repository:', REPO_ROOT)

In [ ]:
# 2. Reviewer configuration. Change only these three values.
REVIEWER_ID = 'A'       # 'A' or 'B'
QUEUE_KIND = 'bbox'     # 'bbox' first, then 'weak'
REVIEWER_CONFIRMED_GUIDE = False

if REVIEWER_ID not in {'A', 'B'}:
    raise ValueError("Set REVIEWER_ID to 'A' or 'B'.")
if QUEUE_KIND not in {'bbox', 'weak'}:
    raise ValueError("Set QUEUE_KIND to 'bbox' or 'weak'.")
if not REVIEWER_CONFIRMED_GUIDE:
    raise RuntimeError(
        'Read docs/DATASET_MANUAL_REVIEW_GUIDE.md, then set '
        'REVIEWER_CONFIRMED_GUIDE = True and rerun this cell.'
    )
print('Reviewer:', REVIEWER_ID, '| queue:', QUEUE_KIND)

In [ ]:
# 3. Locate the verified queues and attached dataset; read train/validation only.
import json
from collections import defaultdict

from audit_gold_existing_data import ImageReader
from validate_kaggle_gold import find_data_source, load_records, ZipDataset
from web_agent.data.review_session import (
    ACTION_LABELS, FAILURE_LABELS, OUTCOME_LABELS, REASON_CODES,
    RECOVERY_ATTEMPTED_VALUES, RECOVERY_LABELS, RECOVERY_SUCCESS_VALUES,
    append_review_event,
    build_review_event, completion_summary, latest_events,
    load_queue_csv, load_review_events, review_targets,
)

INPUT_ROOT = Path('/kaggle/input')
bbox_matches = sorted(INPUT_ROOT.rglob('bbox_review_queue.csv'))
queue_dirs = [path.parent for path in bbox_matches if (path.parent / 'weak_class_review_queue.csv').is_file()]
if len(queue_dirs) != 1:
    raise FileNotFoundError(
        'Attach exactly one output of web-gold-existing-data-improvement; '
        f'compatible queue directories found: {queue_dirs}'
    )
QUEUE_DIR = queue_dirs[0]
BBOX_ROWS = load_queue_csv(QUEUE_DIR / 'bbox_review_queue.csv')
WEAK_ROWS = load_queue_csv(QUEUE_DIR / 'weak_class_review_queue.csv')
QUEUE_ROWS = list(BBOX_ROWS if QUEUE_KIND == 'bbox' else WEAK_ROWS)
secondary_name = f'{QUEUE_KIND}_secondary_review_queue.csv'
secondary_matches = sorted(INPUT_ROOT.rglob(secondary_name))
if len(secondary_matches) > 1:
    raise RuntimeError(
        f'Attach at most one {secondary_name}; found: {secondary_matches}'
    )
if secondary_matches:
    secondary_rows = load_queue_csv(secondary_matches[0])
    QUEUE_ROWS.extend(secondary_rows)
    print('Added secondary-review targets:', len(secondary_rows))
TARGETS = review_targets(QUEUE_ROWS, queue_kind=QUEUE_KIND, reviewer_id=REVIEWER_ID)

KNOWN_DATASET_ROOT = Path('/kaggle/input/datasets/kiyasmahmud/web-gold-40k')
DATA_SOURCE = find_data_source(KNOWN_DATASET_ROOT if KNOWN_DATASET_ROOT.is_dir() else INPUT_ROOT)
IMAGE_READER = ImageReader(DATA_SOURCE)
for target in TARGETS:
    if not target.get('image_width') or not target.get('image_height'):
        image_info, _ = IMAGE_READER.info(str(target.get('state_before') or ''))
        if image_info:
            target['image_width'], target['image_height'] = image_info
SOURCE_ROWS = {
    'train': load_records(DATA_SOURCE, 'split_train.json'),
    'val': load_records(DATA_SOURCE, 'split_val.json'),
}

def row_view(record):
    if 'inputs' in record and 'labels' in record:
        return record['inputs'], record['labels'], record.get('meta', {})
    return record, record, record

SOURCE_BY_SAMPLE = {}
SOURCE_BY_TASK = defaultdict(list)
for split, rows in SOURCE_ROWS.items():
    for position, record in enumerate(rows):
        _, _, meta = row_view(record)
        task_id = str(meta.get('task_id') or meta.get('original_task_id') or f'{split}:{position}')
        step = int(meta.get('step_index', meta.get('step', position)))
        sample_id = str(meta.get('sample_id') or f'{task_id}:{step}')
        SOURCE_BY_SAMPLE[sample_id] = record
        SOURCE_BY_TASK[task_id].append((step, record))
for task_id in SOURCE_BY_TASK:
    SOURCE_BY_TASK[task_id].sort(key=lambda item: item[0])

LOG_PATH = Path(f'/kaggle/working/reviewer_{REVIEWER_ID}_{QUEUE_KIND}_review_events.csv')
resume_matches = sorted(INPUT_ROOT.rglob(LOG_PATH.name))
if len(resume_matches) > 1:
    raise RuntimeError(
        f'Attach at most one prior {LOG_PATH.name}; found: {resume_matches}'
    )
if resume_matches and not LOG_PATH.is_file():
    import shutil
    shutil.copy2(resume_matches[0], LOG_PATH)
    load_review_events(LOG_PATH)  # Validate its immutable-event schema now.
    print('Restored prior event log:', resume_matches[0])
print('Queue source:', QUEUE_DIR)
print('Assigned decision targets:', len(TARGETS))
print('Immutable event log:', LOG_PATH)
print('TEST ROWS READ: 0')

In [ ]:
# 4. Launch the interactive review application.
import io
import math

import ipywidgets as widgets
from IPython.display import Markdown, display, clear_output
from PIL import ImageDraw

def image_widget(reference, *, bbox=None, border='1px solid #bbb'):
    image = IMAGE_READER.image(str(reference or ''))
    if image is None:
        return widgets.HTML('<b style="color:red">Image unavailable</b>')
    image.thumbnail((720, 405))
    if bbox:
        source_info, _ = IMAGE_READER.info(str(reference or ''))
        if source_info:
            source_width, source_height = source_info
            scale_x = image.width / source_width
            scale_y = image.height / source_height
            draw = ImageDraw.Draw(image)
            try:
                x, y, width, height = (float(bbox[name]) for name in ('x', 'y', 'width', 'height'))
                draw.rectangle(
                    (x * scale_x, y * scale_y, (x + width) * scale_x, (y + height) * scale_y),
                    outline='red', width=3,
                )
            except (TypeError, ValueError, KeyError):
                pass
    payload = io.BytesIO()
    image.save(payload, format='PNG')
    return widgets.Image(value=payload.getvalue(), format='png', layout=widgets.Layout(width='49%', border=border))

def source_identity(record, fallback=0):
    _, _, meta = row_view(record)
    task_id = str(meta.get('task_id') or meta.get('original_task_id') or '')
    step = int(meta.get('step_index', meta.get('step', fallback)))
    sample_id = str(meta.get('sample_id') or f'{task_id}:{step}')
    return sample_id, task_id, step

def trajectory_accordion(task_id):
    panels = []
    titles = []
    for fallback, (_, record) in enumerate(SOURCE_BY_TASK.get(task_id, [])):
        inputs, labels, _ = row_view(record)
        sample_id, _, step = source_identity(record, fallback)
        pair = widgets.HBox([
            image_widget(inputs.get('state_before')),
            image_widget(inputs.get('state_after')),
        ])
        caption = widgets.HTML(
            f"<b>{sample_id}</b> | action={labels.get('action_type')} | "
            f"outcome={labels.get('outcome_label')} | failure={labels.get('failure_type_4')} | "
            f"recovery={labels.get('recovery_strategy')} / {labels.get('recovery_success')}"
        )
        panels.append(widgets.VBox([caption, pair]))
        titles.append(f'step {step}: {labels.get("action_type")}')
    accordion = widgets.Accordion(children=panels)
    for index, title in enumerate(titles):
        accordion.set_title(index, title)
    return accordion

index_box = widgets.BoundedIntText(value=1, min=1, max=max(len(TARGETS), 1), description='Item')
previous_button = widgets.Button(description='Previous')
next_button = widgets.Button(description='Next')
next_unreviewed_button = widgets.Button(description='Next unreviewed')
save_button = widgets.Button(description='Save immutable event + next', button_style='success')
reload_button = widgets.Button(description='Reload current')
decision_choices = (
    ('Choose...', ''),
    ('Correct using evidence', 'approve_after_correction'),
    ('Mask bbox; no direct correction evidence', 'mask_bbox_no_evidence'),
    ('Needs discussion', 'needs_discussion'),
    ('Reject/recollect', 'reject_recollect'),
    ('Quarantine for policy', 'quarantine_policy'),
) if QUEUE_KIND == 'bbox' else (
    ('Choose...', ''),
    ('Approve no change', 'approve_no_change'),
    ('Correct using evidence', 'approve_after_correction'),
    ('Needs discussion', 'needs_discussion'),
    ('Reject/recollect', 'reject_recollect'),
    ('Quarantine for policy', 'quarantine_policy'),
)
decision = widgets.Dropdown(options=decision_choices, description='Decision')
reason = widgets.Dropdown(options=REASON_CODES, description='Reason')
evidence = widgets.Text(description='Evidence', placeholder='replay/audit/screenshot reference')
notes = widgets.Textarea(description='Notes', layout=widgets.Layout(width='95%', height='80px'))
check_options = [('', ''), ('Yes', 'yes'), ('No', 'no'), ('Uncertain', 'uncertain'), ('N/A', 'not_applicable')]
action_ok = widgets.Dropdown(options=check_options, description='Action OK')
bbox_ok = widgets.Dropdown(options=check_options, description='BBox OK')
outcome_ok = widgets.Dropdown(options=check_options, description='Outcome OK')
failure_ok = widgets.Dropdown(options=check_options, description='Failure OK')
recovery_ok = widgets.Dropdown(options=check_options, description='Recovery OK')
bbox_values = [widgets.Text(description=name) for name in ('bbox x', 'bbox y', 'bbox width', 'bbox height')]
proposed_action = widgets.Dropdown(options=ACTION_LABELS, description='New action')
proposed_outcome = widgets.Dropdown(options=OUTCOME_LABELS, description='New outcome')
proposed_failure = widgets.Dropdown(options=FAILURE_LABELS, description='New failure')
proposed_attempted = widgets.Dropdown(options=RECOVERY_ATTEMPTED_VALUES, description='New attempted')
proposed_strategy = widgets.Dropdown(options=RECOVERY_LABELS, description='New recovery')
proposed_success = widgets.Dropdown(options=RECOVERY_SUCCESS_VALUES, description='New rec result')
viewer_output = widgets.Output()
status_output = widgets.Output()

def current_target():
    return TARGETS[index_box.value - 1]

def clear_form():
    decision.value = ''
    reason.value = ''
    evidence.value = ''
    notes.value = ''
    for widget in (action_ok, bbox_ok, outcome_ok, failure_ok, recovery_ok):
        widget.value = ''
    for widget in bbox_values:
        widget.value = ''
    for widget in (proposed_action, proposed_outcome, proposed_failure, proposed_attempted, proposed_strategy, proposed_success):
        widget.value = ''

def render_current(*_):
    if not TARGETS:
        with viewer_output:
            clear_output(wait=True)
            display(Markdown('No targets are assigned to this reviewer.'))
        return
    target = current_target()
    events = load_review_events(LOG_PATH)
    prior = latest_events(events, queue_kind=QUEUE_KIND, reviewer_id=REVIEWER_ID).get(target['sample_id'])
    with viewer_output:
        clear_output(wait=True)
        display(Markdown(
            f"## {QUEUE_KIND.upper()} {index_box.value}/{len(TARGETS)} — `{target['sample_id']}`\n"
            f"Task `{target['task_id']}` | split `{target['split']}` | assignment `{target['reviewer_assignment']}`"
        ))
        if prior:
            display(Markdown(f"**Prior saved event:** `{prior['review_decision']}` at {prior['reviewed_at_utc']} (saving again creates a new immutable revision)."))
        if QUEUE_KIND == 'bbox':
            original = json.loads(target['original_bbox'])
            display(Markdown(
                f"Reasons: `{target['geometry_reasons']}`  \n"
                f"Native image: {target['image_width']}×{target['image_height']}  \n"
                f"Original bbox: `{target['original_bbox']}` | action coordinates: `{target['action_coordinates']}`"
            ))
            display(widgets.HBox([
                image_widget(target['state_before'], bbox=original),
                image_widget(target['state_after']),
            ]))
        else:
            display(Markdown(
                f"Focus: `{target['row_focus_targets']}`  \n"
                f"Action `{target['action_type']}` | outcome `{target['outcome_label']}` | "
                f"failure `{target['failure_type_4']}` | recovery `{target['recovery_strategy']}` / `{target['recovery_success']}`"
            ))
            recovery_view = 'RECOVERY:' in str(target.get('row_focus_targets', '')) and target.get('failure_state')
            left = target['failure_state'] if recovery_view else target['state_before']
            right = target['post_recovery_state'] if recovery_view else target['state_after']
            display(Markdown('**Failure state → post-recovery state**' if recovery_view else '**Before → after**'))
            display(widgets.HBox([image_widget(left), image_widget(right)]))
            if recovery_view:
                display(Markdown(
                    f"Executed recovery: `{target['executed_recovery_action']}` value `{target['recovery_action_value']}`"
                ))
        display(Markdown('### Complete trajectory context'))
        display(trajectory_accordion(target['task_id']))

def save_event(_):
    target = current_target()
    with status_output:
        clear_output(wait=True)
        try:
            event = build_review_event(
                target,
                queue_kind=QUEUE_KIND,
                reviewer_id=REVIEWER_ID,
                review_decision=decision.value,
                reason_code=reason.value,
                evidence_reference=evidence.value,
                reviewer_notes=notes.value,
                action_label_ok=action_ok.value,
                bbox_ok=bbox_ok.value,
                outcome_ok=outcome_ok.value,
                failure_type_ok=failure_ok.value,
                recovery_transition_ok=recovery_ok.value,
                proposed_bbox=[widget.value for widget in bbox_values],
                proposed_action_type=proposed_action.value,
                proposed_outcome_label=proposed_outcome.value,
                proposed_failure_type=proposed_failure.value,
                proposed_recovery_attempted=proposed_attempted.value,
                proposed_recovery_strategy=proposed_strategy.value,
                proposed_recovery_success=proposed_success.value,
            )
            append_review_event(LOG_PATH, event)
            print('Saved immutable event:', event['event_id'])
            clear_form()
            move_to_next_unreviewed()
        except Exception as error:
            print('NOT SAVED:', error)

def move(delta):
    def handler(_):
        index_box.value = min(max(index_box.value + delta, 1), len(TARGETS))
        render_current()
    return handler

def move_to_next_unreviewed(_=None):
    completed = latest_events(
        load_review_events(LOG_PATH),
        queue_kind=QUEUE_KIND,
        reviewer_id=REVIEWER_ID,
    )
    start = index_box.value
    ordered = list(range(start, len(TARGETS))) + list(range(0, start))
    for index in ordered:
        if TARGETS[index]['sample_id'] not in completed:
            index_box.value = index + 1
            render_current()
            return
    with status_output:
        clear_output(wait=True)
        print('All assigned targets have at least one saved decision.')

previous_button.on_click(move(-1))
next_button.on_click(move(1))
reload_button.on_click(render_current)
next_unreviewed_button.on_click(move_to_next_unreviewed)
save_button.on_click(save_event)
index_box.observe(render_current, names='value')

correction_box = widgets.VBox([
    widgets.HTML('<b>Proposed correction (leave blank unless evidence proves it)</b>'),
    widgets.HBox(bbox_values[:2]), widgets.HBox(bbox_values[2:]),
    widgets.HBox([proposed_action, proposed_outcome, proposed_failure]),
    widgets.HBox([proposed_attempted, proposed_strategy, proposed_success]),
])
form = widgets.VBox([
    widgets.HBox([previous_button, index_box, next_button, next_unreviewed_button, reload_button]),
    viewer_output,
    widgets.HTML('<h3>Reviewer decision</h3>'),
    widgets.HTML(
        '<small>Approve no change: every check must be Yes or N/A. '
        'Mask/correct a bbox: BBox OK must be No or Uncertain. '
        'Any corrected field needs its matching check set to No or Uncertain.</small>'
    ),
    widgets.HBox([decision, reason]), evidence,
    widgets.HBox([action_ok, bbox_ok, outcome_ok]),
    widgets.HBox([failure_ok, recovery_ok]),
    correction_box, notes, save_button, status_output,
])
display(form)
move_to_next_unreviewed()

In [ ]:
# 5. Progress and safe export. Rerun this cell whenever you want a backup.
import shutil

events = load_review_events(LOG_PATH)
summary = completion_summary(
    TARGETS,
    events,
    queue_kind=QUEUE_KIND,
    reviewer_id=REVIEWER_ID,
)
SUMMARY_PATH = Path(f'/kaggle/working/reviewer_{REVIEWER_ID}_{QUEUE_KIND}_summary.json')
SUMMARY_PATH.write_text(json.dumps(summary, indent=2), encoding='utf-8')
print(json.dumps({
    key: value for key, value in summary.items()
    if key not in {'incomplete_sample_ids', 'unresolved_sample_ids'}
}, indent=2))
print('Event log:', LOG_PATH)
print('Summary:', SUMMARY_PATH)

FINAL_EXPORT = False  # Set True only when remaining_targets is zero.
if FINAL_EXPORT:
    assert summary['remaining_targets'] == 0, (
        f"{summary['remaining_targets']} assigned targets remain"
    )
    EXPORT_DIR = Path(f'/kaggle/working/reviewer_{REVIEWER_ID}_{QUEUE_KIND}_final')
    EXPORT_DIR.mkdir(parents=True, exist_ok=True)
    shutil.copy2(LOG_PATH, EXPORT_DIR / LOG_PATH.name)
    shutil.copy2(SUMMARY_PATH, EXPORT_DIR / SUMMARY_PATH.name)
    archive = shutil.make_archive(str(EXPORT_DIR), 'zip', root_dir=EXPORT_DIR)
    print('FINAL REVIEW EXPORT:', archive)
    if summary['unresolved_targets']:
        print('ADJUDICATION REQUIRED for needs_discussion targets:', summary['unresolved_targets'])
else:
    print('Backup the CSV now. Set FINAL_EXPORT=True only after completing this queue.')

## Required workflow

- Finish `bbox`, export it, then restart the kernel and finish `weak`.
- To resume unfinished work, attach the prior queue-specific event CSV before running cell 3.
- Download the event CSV frequently; Kaggle sessions are temporary.
- Saving a revised decision appends a new event. Earlier decisions are preserved.
- `A+B` rows must appear independently in both reviewers' logs.
- Do not mark the source dataset approved or edit split JSON files from this notebook.
- Send both reviewers' four final files (bbox CSV/summary and weak CSV/summary) to the data lead for agreement and adjudication.